# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [43]:
# Only needed for Udacity workspace

import importlib.util
import sys


In [44]:
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv
import chromadb
from pydantic import BaseModel
from tavily import TavilyClient
import json

In [45]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


In [46]:
chroma_client = chromadb.PersistentClient(path=".data/chromadb")
embedding_fn = chromadb.utils.embedding_functions.OpenAIEmbeddingFunction()
collection = chroma_client.get_collection("games")


#### Retrieve Game Tool

In [47]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# Tool Docstring:
#    

@tool
def retrieve_game(query: str) -> list:
    """Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game"""
    collection = chroma_client.get_collection("games")

    results = collection.query(query_texts=[query], n_results=5)
    return results

#### Evaluate Retrieval Tool

In [48]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str
    
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result"""
    llm = LLM(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
    prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
    Question: {question}
    Retrieved Docs: {retrieved_docs}
    Give a detailed explanation, so it's possible to take an action to accept it or not.
    """
    result = llm.invoke(prompt, response_format=EvaluationReport)
    return result.dict()


#### Game Web Search Tool

In [49]:

# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 
client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def web_search(query: str) -> str:
    """Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry."""
    results = client.search(query)
    formatted = []
    for r in results.get("results", []):
        formatted.append({
            "title": r.get("title", ""),
            "url": r.get("url", ""),
            "content": r.get("content", "")
        })
    output = {"results": formatted}
    return json.dumps(output, indent=2)


### Agent

In [50]:

# Changed to custom agent since I have to ensure certain calls
import uuid
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.memory import ShortTermMemory
from lib.tooling import ToolCall
from openai.types.chat.chat_completion_message_tool_call import Function as ToolCallFunction
from typing import TypedDict, List, Optional, Any


class GameResearchState(TypedDict):
    user_query: str
    instructions: str
    messages: List
    session_id: str
    current_tool_calls: Optional[List]  # tool calls pending execution
    retrieved_docs: Any
    evaluation: Optional[dict]
    web_results: Optional[str]
    total_tokens: int


instructions = """You are UdaPlay, an AI Research Agent for the video game industry.
Answer questions about the videogame industry clearly and accurately.
"""


class GameResearchAgent:
    """
    State machine agent in which it is ensured that RAG content is always evaluated and if not good enough it is ensured web search is triggered.
    """

    def __init__(self, model_name: str, instructions: str, tools: list):
        self.model_name = model_name
        self.instructions = instructions
        self._tools = {t.name: t for t in tools}
        self.memory = ShortTermMemory()
        self.workflow = self._build_workflow()

    def _create_tool_call_message(self, name: str, arguments: dict) -> ToolCall:
        return ToolCall(
            id=f"call_{uuid.uuid4().hex[:24]}",
            function=ToolCallFunction(name=name, arguments=json.dumps(arguments)),
            type="function",
        )

    def _message_prep(self, state: dict) -> dict:
        messages = list(state.get("messages", []))
        if not messages:
            messages = [SystemMessage(content=state["instructions"])]
        return {
            "messages": messages + [UserMessage(content=state["user_query"])],
            "session_id": state["session_id"],
        }

    def _llm_retrieve_step(self, state: dict) -> dict:
        """initial LLM step"""
        llm = LLM(model=self.model_name, tools=[self._tools["retrieve_game"]])
        response = llm.invoke(state["messages"])

        # Fallback: if LLM didn't produce a tool call, synthesise one
        tool_calls = response.tool_calls
        if not tool_calls:
            tool_calls = [self._create_tool_call_message("retrieve_game", {"query": state["user_query"]})]

        tokens = state.get("total_tokens", 0)
        if response.token_usage:
            tokens += response.token_usage.total_tokens

        return {
            "messages": state["messages"] + [AIMessage(content=response.content, tool_calls=tool_calls)],
            "current_tool_calls": tool_calls,
            "total_tokens": tokens,
            "session_id": state["session_id"],
        }

    def _retrieve_step(self, state: dict) -> dict:
        """Execute the retrieve_game tool call and produce a ToolMessage."""
        tool_calls = state.get("current_tool_calls") or []
        new_messages = []
        retrieved_docs = state.get("retrieved_docs")

        for call in tool_calls:
            if call.function.name == "retrieve_game":
                args = json.loads(call.function.arguments)
                retrieved_docs = self._tools["retrieve_game"](**args)
                new_messages.append(ToolMessage(
                    content=json.dumps(str(retrieved_docs)),
                    tool_call_id=call.id,
                    name="retrieve_game",
                ))

        return {
            "retrieved_docs": retrieved_docs,
            "messages": state["messages"] + new_messages,
            "current_tool_calls": None,
            "session_id": state["session_id"],
        }

    def _evaluate_step(self, state: dict) -> dict:
        """
        ensure evaluate is called in its own step
        """
        args = {"question": state["user_query"], "retrieved_docs": state["retrieved_docs"]}
        call = self._create_tool_call_message("evaluate_retrieval", args)

        ai_msg = AIMessage(content=None, tool_calls=[call])

        raw = self._tools["evaluate_retrieval"](**args)

        evaluation = raw if "useful" in raw else json.loads(raw.get("content", "{}"))

        tool_msg = ToolMessage(
            content=json.dumps(evaluation),
            tool_call_id=call.id,
            name="evaluate_retrieval",
        )

        return {
            "evaluation": evaluation,
            "messages": state["messages"] + [ai_msg, tool_msg],
            "session_id": state["session_id"],
        }

    def _web_search_step(self, state: dict) -> dict:
        """
        Webseach if RAG is not useful. 
        """
        args = {"query": state["user_query"]}
        call = self._create_tool_call_message("web_search", args)

        ai_msg = AIMessage(content=None, tool_calls=[call])

        result = self._tools["web_search"](**args)

        tool_msg = ToolMessage(
            content=json.dumps(result),
            tool_call_id=call.id,
            name="web_search",
        )

        return {
            "web_results": result,
            "messages": state["messages"] + [ai_msg, tool_msg],
            "session_id": state["session_id"],
        }

    def _final_answer_step(self, state: dict) -> dict:
        """Real LLM call — synthesises the final answer from the full message chain."""
        if state.get("web_results"):
            answer_prompt = (
                f"Based on everything retrieved above, provide your final answer to: "
                f"{state['user_query']!r}\n\n"
                "Web search was used. You MUST end your answer with a Citations section "
                "listing the source URLs from the search results above:\n\n"
                "**Citations:**\n- [Title](URL)\n\n"
                "Include at least 1–2 source URLs. Do NOT invent URLs."
            )
        else:
            answer_prompt = (
                f"Based on everything retrieved above, provide your final answer to: "
                f"{state['user_query']!r}\n\n"
                "Answer using only the retrieved information. Do NOT include any citations or URLs."
            )

        answer_request = UserMessage(content=answer_prompt)

        llm = LLM(model=self.model_name)
        response = llm.invoke(state["messages"] + [answer_request])
        ai_msg = AIMessage(content=response.content)

        tokens = state.get("total_tokens", 0)
        if response.token_usage:
            tokens += response.token_usage.total_tokens

        return {
            "messages": state["messages"] + [ai_msg],
            "total_tokens": tokens,
            "session_id": state["session_id"],
        }


    def _build_workflow(self):
        machine = StateMachine[GameResearchState](GameResearchState)

        entry      = EntryPoint[GameResearchState]()
        msg_prep   = Step[GameResearchState]("message_prep", self._message_prep)
        llm_call   = Step[GameResearchState]("llm_retrieve", self._llm_retrieve_step)
        retrieve   = Step[GameResearchState]("retrieve_game", self._retrieve_step)
        evaluate   = Step[GameResearchState]("evaluate_retrieval", self._evaluate_step)
        web        = Step[GameResearchState]("web_search", self._web_search_step)
        answer     = Step[GameResearchState]("final_answer", self._final_answer_step)
        end        = Termination[GameResearchState]()

        machine.add_steps([entry, msg_prep, llm_call, retrieve, evaluate, web, answer, end])

        machine.connect(entry, msg_prep)
        machine.connect(msg_prep, llm_call)
        machine.connect(llm_call, retrieve)
        machine.connect(retrieve, evaluate)  

        def route_after_evaluate(state):
            """Branch on evaluation result — skip web_search when docs are sufficient."""
            return answer if state["evaluation"]["useful"] else web

        machine.connect(evaluate, [web, answer], route_after_evaluate)
        machine.connect(web,    answer)
        machine.connect(answer, end)

        return machine

    def invoke(self, query: str, session_id: str = None):
        session_id = session_id or "default"
        self.memory.create_session(session_id)

        previous_messages = []
        last_run = self.memory.get_last_object(session_id)
        if last_run:
            last_state = last_run.get_final_state()
            if last_state:
                previous_messages = last_state["messages"]


        initial: GameResearchState = {
            "user_query": query,
            "instructions": self.instructions,
            "messages": previous_messages,
            "session_id": session_id,
            "current_tool_calls": None,
            "retrieved_docs": None,
            "evaluation": None,
            "web_results": None,
            "total_tokens": 0,
        }

        run = self.workflow.run(initial)
        self.memory.add(run, session_id)
        return run


agent = GameResearchAgent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, web_search],
)


In [51]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
for i, question in enumerate([
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?"
]):
    print("\n\n")
    print("=" * 80)
    print(f"Question {i+1}: {question}")
    run = agent.invoke(question, session_id=f"session_{i+1}")
    print(f"Answer {i+1}: {run.get_final_state()["messages"][-1].content}")

    for j, msg in enumerate(run.get_final_state()["messages"]):
        print(f"\nMessage {j+1}: {msg}")

    print(f"\nfull trace: {run.get_final_state()}")




Question 1: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_retrieve
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: final_answer
[StateMachine] Terminating: __termination__
Answer 1: Pokémon Gold and Silver were released in 1999 for the Game Boy Color.

Message 1: role='system' content='You are UdaPlay, an AI Research Agent for the video game industry.\nAnswer questions about the videogame industry clearly and accurately.\n'

Message 2: role='user' content='When Pokémon Gold and Silver was released?'

Message 3: role='assistant' content=None tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_MdvYPNN9NKtjypNeR9MKfXJx', function=Function(arguments='{"query":"Pokémon Gold and Silver"}', name='retrieve_game'), type='function')] token_usage=None

Message 4: role='tool' content='"{\'ids\': [[\'00

### (Optional) Advanced